In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import altair as alt

In [4]:
import itertools

In [5]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [6]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [7]:
lah = pd.read_excel(r"C:\Users\HP\Project-IDS-6C-AQI-MoizAhmed(150)-AbdulRauf(123)\IDS-6C-MoizAhmed-(BSE-23F-150)-AbdulRauf(BSE-23F-123)\Ds project\dataset\pakistan_aqi_weather.xlsx", sheet_name = "Lahore")

In [8]:
print(lah["time"].dtype)

datetime64[ns]


In [9]:
lah.set_index('time', inplace=True)

In [10]:
lah = lah[[
    'pm2_5',
    'temperature_2m',
    'relative_humidity_2m',
    'pressure_msl',
    'wind_speed_10m',
    'rain',
    'cloud_cover'
]]

In [11]:
lah.fillna(method='ffill', inplace=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_1276\2110886799.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lah.fillna(method='ffill', inplace=True)


In [12]:
lah_monthly = lah.resample('M').mean()

C:\Users\HP\AppData\Local\Temp\ipykernel_1276\1281214369.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  lah_monthly = lah.resample('M').mean()


In [13]:
lah_reset = lah_monthly.reset_index()

In [14]:
print(lah_reset.head())

        time       pm2_5  temperature_2m  relative_humidity_2m  pressure_msl  \
0 2022-08-31   47.886357       29.522789             75.592204   1001.525787   
1 2022-09-30   67.801944       29.035417             69.394444   1004.464444   
2 2022-10-31  102.928360       24.433065             61.327957   1011.063038   
3 2022-11-30  103.447639       18.043472             70.244444   1015.042222   
4 2022-12-31  123.179435       13.740323             73.432796   1016.464651   

   wind_speed_10m      rain  cloud_cover  
0        7.732084  0.148726    42.047976  
1        6.300972  0.071667    17.284722  
2        7.367204  0.019220     3.735215  
3        7.839167  0.028889    22.322222  
4        6.838441  0.008871    24.250000  


In [15]:
alt.data_transformers.disable_max_rows()


chart = alt.Chart(lah_reset).mark_line().encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5'),
    tooltip=['Date:T', 'pm2_5:Q']
).properties(
    title='4-Year PM2.5 Trend',
    width=800,
    height=400
).interactive()

chart

alt.Chart(...)

In [16]:
train_size = int(len(lah) * 0.78)   # ~36 months 

train = lah.iloc[:train_size]
test = lah.iloc[train_size:]

In [17]:
train_monthly = train.resample('ME').mean()

In [18]:
train_monthly_plot = train_monthly.reset_index()

In [19]:
train_chart = alt.Chart(train_monthly_plot).mark_line(color = 'red').encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5'),
    tooltip=['index:T', 'pm2_5:Q']
).properties(
    title='AQI - Training Data (36 Months)',
    width=800,
    height=400
).interactive()

train_chart

alt.Chart(...)

In [20]:
print("Full range:", lah.index.min(), "to", lah.index.max())
print("Train range:", train.index.min(), "to", train.index.max())
print("Test range:", test.index.min(), "to", test.index.max())

Full range: 2022-08-04 05:00:00 to 2026-05-18 23:00:00
Train range: 2022-08-04 05:00:00 to 2025-07-18 12:00:00
Test range: 2025-07-18 13:00:00 to 2026-05-18 23:00:00


In [21]:
test_monthly = test.resample('ME').mean()

In [22]:
train_monthly_plot = test_monthly.reset_index()

In [23]:
test_chart = alt.Chart(train_monthly_plot).mark_line(
    color='red'
).encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5'),
    tooltip=['index:T', 'pm2_5:Q']
).properties(
    title='AQI - Test Data (10 Months)',
    width=800,
    height=400
).interactive()

test_chart

alt.Chart(...)

In [24]:
lah_monthly = lah.resample('ME').mean()

In [25]:
train = lah_monthly.iloc[:36]
test = lah_monthly.iloc[36:]

In [26]:
y_train = train['pm2_5']
X_train = train[['temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'wind_speed_10m', 'rain', 'cloud_cover']]

y_test = test['pm2_5']
X_test = test[['temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'wind_speed_10m', 'rain', 'cloud_cover']]


In [27]:
model = SARIMAX(y_train,
                exog=X_train,
                order=(1,2,1),
                seasonal_order=(2,0,1,12),
                enforce_stationarity=False,
                enforce_invertibility=False)
results = model.fit()

C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [28]:
forecast = results.forecast(steps=len(test), exog=X_test)

In [29]:
rmse = np.sqrt(mean_squared_error(y_test, forecast))
mae = mean_absolute_error(y_test, forecast)
mape = np.mean(np.abs((y_test - forecast) / y_test)) * 100
model_accuracy = 100 - mape

In [30]:
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"Model Accuracy{model_accuracy:.2f}%")

RMSE: 8.46
MAE: 7.41
MAPE: 12.45%
Model Accuracy87.55%


In [31]:
actual_df = lah_monthly['pm2_5'].reset_index()
actual_df['Type'] = 'Actual Data'

In [32]:
forecast_df = pd.DataFrame({
    'time': test.index,
    'pm2_5': forecast.values,
    'Type': 'Predicted (SARIMAX with Weather)'
})

In [33]:
plot_df = pd.concat([actual_df, forecast_df], ignore_index=True)

In [34]:
alt.data_transformers.disable_max_rows()

chart = alt.Chart(plot_df).mark_line().encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5 Level'),
    color=alt.Color('Type:N', title='Legend', scale=alt.Scale(
        domain=['Actual Data', 'Predicted (SARIMAX with Weather)'],
        range=['#1f77b4', '#d62728'] # Blue for real data, Red for prediction
    )),
    strokeDash=alt.StrokeDash('Type:N', scale=alt.Scale(
        domain=['Actual Data', 'Predicted (SARIMAX with Weather)'],
        range=[[0], [4, 4]] # Solid line for actuals, dashed line for predictions
    )),
    tooltip=[alt.Tooltip('time:T', title='Date'), alt.Tooltip('pm2_5:Q', title='PM2.5')]
).properties(
    title='Accurate 10-Month PM2.5 Forecast using Weather Features',
    width=800,
    height=400
).interactive()

chart

alt.Chart(...)

In [35]:
test_df = pd.DataFrame({
    'time': test.index,
    'pm2_5': np.asarray(test['pm2_5'] if 'pm2_5' in test else test).flatten(),
    'Type': 'Actual Test Data'
})

# 2. Structure the forecast data safely using .flatten()
forecast_df = pd.DataFrame({
    'time': test.index,
    'pm2_5': np.asarray(forecast).flatten(),
    'Type': 'Predicted Values'
})

# 3. Combine them into a single long-format DataFrame
plot_df = pd.concat([test_df, forecast_df], ignore_index=True)

# 4. Generate the Test vs Prediction Altair Chart
alt.data_transformers.disable_max_rows()

chart = alt.Chart(plot_df).mark_line(point=True).encode(
    x=alt.X('time:T', title='Date'),
    y=alt.Y('pm2_5:Q', title='PM2.5 Level'),
    color=alt.Color('Type:N', title='Legend', scale=alt.Scale(
        domain=['Actual Test Data', 'Predicted Values'],
        range=['#2ca02c', '#d62728']  # Green for Actual Test, Red for Predictions
    )),
    strokeDash=alt.StrokeDash('Type:N', scale=alt.Scale(
        domain=['Actual Test Data', 'Predicted Values'],
        range=[[0], [4, 4]]  # Solid line for actual test, dashed line for predictions
    )),
    tooltip=[alt.Tooltip('time:T', title='Date'), alt.Tooltip('pm2_5:Q', title='PM2.5')]
).properties(
    title='10-Month PM2.5: Actual Test Data vs Predicted Values',
    width=800,
    height=400
).interactive()

chart

alt.Chart(...)